# The transformer decoder
1. The data: one real English-French pair, shifted right.
2. One decoder block with the paper's sizes (d_model = 512, 8 heads, d_ff = 2048): shapes and parameter counts.
3. The pair traced through the input steps, 6 decoder blocks and the output layer.
4. Training: one pass gives a prediction for every position; the loss before training.
5. The whole base model's parameter count.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import re, random, collections
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

np.set_printoptions(precision=3, suppress=True)
tf.config.experimental.enable_tensor_float_32_execution(False)   # exact float32 on the GPU


def positional_encoding(n_pos, d_model):
    """PE[pos, 2i] = sin(pos / 10000^(2i/d)), PE[pos, 2i+1] = cos(same angle) (Vaswani et al. 2017, 3.5)."""
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d_model // 2)[None, :]
    angle = pos / 10000 ** (2 * i / d_model)
    pe = np.zeros((n_pos, d_model))
    pe[:, 0::2] = np.sin(angle)
    pe[:, 1::2] = np.cos(angle)
    return pe.astype("float32")


class EncoderBlock(keras.layers.Layer):
    """Multi-head attention -> add & norm -> feed-forward (ReLU) -> add & norm (Vaswani et al. 2017, 3.1)."""
    def __init__(self, d_model, heads, d_ff, **kw):
        super().__init__(**kw)
        self.mha = keras.layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads)
        self.norm1 = keras.layers.LayerNormalization()
        self.ffn1 = keras.layers.Dense(d_ff, activation="relu")
        self.ffn2 = keras.layers.Dense(d_model)
        self.norm2 = keras.layers.LayerNormalization()

    def call(self, x):
        z_norm = self.norm1(x + self.mha(x, x))
        return self.norm2(z_norm + self.ffn2(self.ffn1(z_norm)))


class DecoderBlock(keras.layers.Layer):
    """Masked multi-head attention -> add & norm -> cross-attention -> add & norm -> feed-forward -> add & norm
    (Vaswani et al. 2017, 3.1). Post-norm, as in the paper."""
    def __init__(self, d_model, heads, d_ff, **kw):
        super().__init__(**kw)
        self.self_mha = keras.layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads)
        self.norm1 = keras.layers.LayerNormalization()
        self.cross_mha = keras.layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads)
        self.norm2 = keras.layers.LayerNormalization()
        self.ffn1 = keras.layers.Dense(d_ff, activation="relu")
        self.ffn2 = keras.layers.Dense(d_model)
        self.norm3 = keras.layers.LayerNormalization()

    def call(self, x, enc):
        z, w_self = self.self_mha(x, x, use_causal_mask=True, return_attention_scores=True)
        z_norm = self.norm1(x + z)                                    # add & norm 1
        zc, w_cross = self.cross_mha(z_norm, enc, return_attention_scores=True)   # queries: decoder; keys, values: encoder
        zc_norm = self.norm2(z_norm + zc)                             # add & norm 2
        hidden = self.ffn1(zc_norm)
        y = self.ffn2(hidden)
        y_norm = self.norm3(zc_norm + y)                              # add & norm 3
        self.trace = dict(z=z, z_norm=z_norm, w_self=w_self, zc=zc, zc_norm=zc_norm, w_cross=w_cross,
                          hidden=hidden, y=y, y_norm=y_norm)     # kept for inspection
        return y_norm

## 1. The data and the shift to the right
The English-French corpus of the Keras examples, tokenised and filtered as in the encoder-decoder Note
(English up to 8 words, French up to 10), with the same 40,000 training pairs and the same French vocabulary:
the 8,000 most frequent words plus <pad>, <unk>, <start>, <end>.

In [2]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = open(os.path.join(path, "fra.txt"), encoding="utf8").read().strip().split("\n")
tok = lambda s: re.findall(r"[\w']+|[^\w\s]", s.lower())
groups = collections.defaultdict(list)
for line in lines:
    e, f = line.split("\t")[:2]
    e, f = tok(e), tok(f)
    if len(e) <= 8 and len(f) <= 10:
        groups[tuple(e)].append(f)
keys = sorted(groups)
random.Random(0).shuffle(keys)
train = [(list(e), f) for e in keys[1500:] for f in groups[e]]
random.Random(1).shuffle(train)
train = train[:40000]


def vocab(seqs, n):
    c = collections.Counter(w for s in seqs for w in s)
    words = ["<pad>", "<unk>", "<start>", "<end>"] + [w for w, _ in c.most_common(n)]
    return {w: i for i, w in enumerate(words)}, words


en_ix, en_w = vocab([e for e, _ in train], 6000)
fr_ix, fr_w = vocab([f for _, f in train], 8000)
V = len(fr_w)
print("French vocabulary V =", V, "  English vocabulary =", len(en_w))

english = tok("We're friends.")
french = tok("Nous sommes amis.")
print("pair in the file:", ("We're friends.\tNous sommes amis." in lines), english, "->", french)
dec_in = ["<start>"] + french                 # shifted right: <start> in front
dec_out = french + ["<end>"]                  # the target: the same words, one step ahead, <end> at the back
in_ids = [fr_ix[w] for w in dec_in]
out_ids = [fr_ix[w] for w in dec_out]
en_ids = [en_ix[w] for w in english]
print(pd.DataFrame({"position": range(1, 6), "decoder input": dec_in, "input id": in_ids,
                    "target": dec_out, "target id": out_ids}).to_string(index=False))

French vocabulary V = 8004   English vocabulary = 6004
pair in the file: True ["we're", 'friends', '.'] -> ['nous', 'sommes', 'amis', '.']
 position decoder input  input id target  target id
        1       <start>         2   nous         21
        2          nous        21 sommes         85
        3        sommes        85   amis        250
        4          amis       250      .          4
        5             .         4  <end>          3


## 2. One decoder block with the paper's sizes
Formula: two attention layers of 4d^2 + 4d each, the feed-forward network d*d_ff + d_ff + d_ff*d + d,
three layer norms of 2d each.

In [3]:
d, h, d_ff = 512, 8, 2048
keras.utils.set_random_seed(0)
n_en, n_fr = len(english), len(dec_in)
enc_out_demo = np.random.default_rng(0).normal(size=(1, n_en, d)).astype("float32")
x_demo = np.random.default_rng(1).normal(size=(1, n_fr, d)).astype("float32")
block = DecoderBlock(d, h, d_ff)
block(x_demo, enc_out_demo)

mha = 4 * d * d + 4 * d
ffn = d * d_ff + d_ff + d_ff * d + d
norms = 3 * 2 * d
formula_block = 2 * mha + ffn + norms
parts = pd.DataFrame([
    ("masked multi-head attention", mha, block.self_mha.count_params()),
    ("cross-attention", mha, block.cross_mha.count_params()),
    ("feed-forward network", ffn, block.ffn1.count_params() + block.ffn2.count_params()),
    ("three layer norms", norms, block.norm1.count_params() + block.norm2.count_params() + block.norm3.count_params()),
    ("one decoder block", formula_block, block.count_params()),
], columns=["part", "formula", "keras"])
print(parts.to_string(index=False))
assert (parts.formula == parts.keras).all()
enc_block = EncoderBlock(d, h, d_ff)
enc_block(enc_out_demo)
print("one encoder block, for comparison:", enc_block.count_params())
print("6 decoder blocks:", 6 * formula_block, "  6 encoder blocks:", 6 * enc_block.count_params())

                       part  formula   keras
masked multi-head attention  1050624 1050624
            cross-attention  1050624 1050624
       feed-forward network  2099712 2099712
          three layer norms     3072    3072
          one decoder block  4204032 4204032
one encoder block, for comparison: 3152384
6 decoder blocks: 25224192   6 encoder blocks: 18914304


## 3. The pair traced through the decoder
Encoder: "we're friends ." through an untrained 6-block encoder (its output H_enc is 3 x 512).
Decoder input: embedding (times sqrt(d_model), section 3.4 of the paper) + positional encoding, then 6 decoder
blocks, then the linear layer (512 -> V) and the softmax. All weights are untrained: only the shapes and the
zeros of the mask carry meaning here.

In [4]:
keras.utils.set_random_seed(0)
emb_en = keras.layers.Embedding(len(en_w), d)
emb_fr = keras.layers.Embedding(V, d)
encoder = [EncoderBlock(d, h, d_ff) for _ in range(6)]
decoder = [DecoderBlock(d, h, d_ff) for _ in range(6)]
final = keras.layers.Dense(V)                  # the linear layer; softmax follows

x = emb_en(np.array([en_ids])) * np.sqrt(d) + positional_encoding(n_en, d)
for b in encoder:
    x = b(x)
H_enc = x
print("encoder output H_enc:", H_enc.shape[1:])

E = emb_fr(np.array([in_ids]))
X = E * np.sqrt(d) + positional_encoding(n_fr, d)
print("decoder input ids:", in_ids, " embeddings:", E.shape[1:], " X:", X.shape[1:])
y = X
for b in decoder:
    y = b(y, H_enc)
keep = decoder[0].trace
for k in ("z", "z_norm", "w_self", "zc", "zc_norm", "w_cross", "hidden", "y", "y_norm"):
    print(f"block 1 {k:8s}", tuple(keep[k].shape[1:]))
print("output of block 6:", y.shape[1:])
logits = final(y)
probs = keras.ops.softmax(logits, axis=-1).numpy()[0]
print("logits:", logits.shape[1:], " probabilities:", probs.shape, " row sums:", probs.sum(-1).round(6))
print("output layer parameters: 512 x V + V =", d * V + V, "  keras:", final.count_params())
assert final.count_params() == d * V + V

w_self = keep["w_self"].numpy()[0].mean(0)     # average over the 8 heads
w_cross = keep["w_cross"].numpy()[0].mean(0)
print("masked self-attention weights, block 1 (mean of 8 heads), rows = queries:\n", w_self)
print("upper triangle exactly 0:", np.all(np.triu(keep["w_self"].numpy()[0], 1) == 0))
print("cross-attention weights, block 1 (mean of 8 heads), rows = French positions, columns = English words:\n", w_cross)
pd.DataFrame(w_cross, index=dec_in, columns=english).to_csv("data/untrained_cross_weights.csv")

encoder output H_enc: (3, 512)
decoder input ids: [2, 21, 85, 250, 4]  embeddings: (5, 512)  X: (5, 512)


block 1 z        (5, 512)
block 1 z_norm   (5, 512)
block 1 w_self   (8, 5, 5)
block 1 zc       (5, 512)
block 1 zc_norm  (5, 512)
block 1 w_cross  (8, 5, 3)
block 1 hidden   (5, 2048)
block 1 y        (5, 512)
block 1 y_norm   (5, 512)
output of block 6: (5, 512)
logits: (5, 8004)  probabilities: (5, 8004)  row sums: [1. 1. 1. 1. 1.]
output layer parameters: 512 x V + V = 4106052   keras: 4106052
masked self-attention weights, block 1 (mean of 8 heads), rows = queries:
 [[1.    0.    0.    0.    0.   ]
 [0.485 0.515 0.    0.    0.   ]
 [0.36  0.288 0.352 0.    0.   ]
 [0.258 0.239 0.259 0.243 0.   ]
 [0.225 0.17  0.168 0.158 0.28 ]]
upper triangle exactly 0: True
cross-attention weights, block 1 (mean of 8 heads), rows = French positions, columns = English words:
 [[0.332 0.336 0.332]
 [0.333 0.342 0.325]
 [0.327 0.354 0.319]
 [0.326 0.352 0.322]
 [0.319 0.351 0.33 ]]


## 4. Training: every position at once
One forward pass gives 5 probability rows, one per position. The loss compares each row with the target word of
that position (cross-entropy, averaged). Before training the model has no preference, so each correct word gets
about 1/V of the probability and the loss is close to ln V.

In [5]:
p_gold = probs[np.arange(n_fr), out_ids]
print("probability given to each target word:", p_gold, " times V (1 = an even share):", (p_gold * V).round(2))
loss = float(-np.log(p_gold).mean())
keras_loss = float(keras.losses.SparseCategoricalCrossentropy(from_logits=True)(np.array([out_ids]), logits))
print("loss by hand:", round(loss, 3), " keras:", round(keras_loss, 3), " ln V:", round(float(np.log(V)), 3))
assert abs(loss - keras_loss) < 1e-4
top = probs.argmax(-1)
print("most likely word per position before training:", [fr_w[k] for k in top])

probability given to each target word: [0. 0. 0. 0. 0.]  times V (1 = an even share): [0.76 0.57 0.79 0.38 1.04]
loss by hand: 9.386  keras: 9.386  ln V: 8.988
most likely word per position before training: ['mit', 'mit', 'mit', 'mit', 'mit']


## 5. The output layer on a toy vocabulary
A worked example of linear + softmax with 4 words.

In [6]:
toy_words = ["nous", "sommes", "amis", "<end>"]
toy_logits = np.array([2.0, 1.0, 0.5, -1.0])
toy_p = np.exp(toy_logits) / np.exp(toy_logits).sum()
print({w: round(float(p), 3) for w, p in zip(toy_words, toy_p)}, " sum", round(float(toy_p.sum()), 6))

{'nous': 0.609, 'sommes': 0.224, 'amis': 0.136, '<end>': 0.03}  sum 1.0


## 6. The whole base model
Vaswani et al. (2017) share one embedding matrix between the encoder input, the decoder input and the
pre-softmax linear layer (3.4), with a shared English-German vocabulary of about 37,000 tokens (5.1).

In [7]:
v_shared = 37000
total = 6 * enc_block.count_params() + 6 * formula_block + v_shared * d
rows = [("6 encoder blocks", 6 * enc_block.count_params()), ("6 decoder blocks", 6 * formula_block),
        ("shared embedding (37,000 x 512)", v_shared * d)]
pd.DataFrame(rows + [("total", total)], columns=["part", "parameters"]).to_csv("data/param_counts.csv", index=False)
for r in rows + [("total", total)]:
    print(f"{r[0]:35s} {r[1]:>12,}")

6 encoder blocks                      18,914,304
6 decoder blocks                      25,224,192
shared embedding (37,000 x 512)       18,944,000
total                                 63,082,496
